# Retrieval Project Submission Notebook

Notebook for final submission generation. Main pipeline stages are visible here, while implementation stays in `src/`.


## Pipeline Overview

This notebook keeps the **main submission flow visible** while delegating implementation to `src/`.

Pipeline stages:
1. detect the runtime and locate the project
2. load and preprocess documents and queries
3. prepare the configured first-stage retriever
4. optionally predict categories for queries
5. optionally build the cross-encoder reranker
6. run first-stage retrieval on the test queries
7. optionally rerank the top candidates
8. write the final Kaggle submission file


In [1]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

import pandas as pd

from src.infra.notebook import setup_notebook
from src.config import DEFAULT_CONFIG
from src.evaluation import load_ground_truth
from src.pipeline import (
    bootstrap,
    build_cross_encoder_reranker,
    load_project_frames,
    predict_categories,
    prepare_retrievers,
    rerank_retrieval_results,
    run_first_stage_retrieval,
    write_submission,
)

runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')

paths, config = bootstrap()
frames = load_project_frames(paths, DEFAULT_CONFIG)
ground_truth = load_ground_truth(paths.data_dir / 'qgts_train.json')

print(f'Runtime environment: {paths.runtime_env}')
print(f'Project directory  : {paths.project_dir}')
print(f'Data directory     : {paths.data_dir}')
print(f'Documents          : {len(frames.docs):,}')
print(f'Train queries      : {len(frames.train_queries):,}')
print(f'Test queries       : {len(frames.test_queries):,}')


Detected runtime: local
Project root    : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project
Runtime environment: local
Project directory  : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project
Data directory     : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/data
Documents          : 216,041
Train queries      : 327
Test queries       : 141


## Effective Config Summary

This cell prints the exact submission settings used by the notebook so the executed output is self-describing.

In [2]:
print(f"Final model              : {config.retrieval_pipeline.final_model}")
print(f"Submit top_k             : {config.retrieval_pipeline.submit_top_k:,}")
print(f"Rerank top_m             : {config.cross_encoder.rerank_top_m}")
print(f"Category bonus           : {config.cross_encoder.category_bonus:.2f}")
print(f"Classifier query tags on : {config.data_columns.use_query_tags_in_classifier}")


Final model              : embedding
Submit top_k             : 12,500
Rerank top_m             : 45
Category bonus           : 0.50
Classifier query tags on : True


## Step 1: Retriever Preparation

The first-stage retriever is prepared here. Depending on the config, this can be TF-IDF, BM25, or embedding-based retrieval.
Artifacts are built once and cached under `cache/` so repeated notebook runs do not recompute them unnecessarily.


In [3]:
prepared_retrievers = prepare_retrievers(frames, paths, config=config)
print(f'Prepared retrievers: {sorted(prepared_retrievers.keys())}')


Loading model weights from cache: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/cache/sentence_transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_6485efc872cab480.npy
Prepared retrievers: ['embedding']


## Step 2: Category Prediction

If category filtering is enabled, the notebook trains or loads the lightweight category classifier, predicts categories for train and test queries, and builds a document-category lookup.
This category information is later used either to filter retrieval or to apply a soft bonus during reranking.


In [4]:
category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=config)
if category_artifacts.classifier_artifacts is None:
    print('Category filtering disabled in config.')
else:
    print(f'Train category predictions: {len(category_artifacts.train_query_category_map):,}')
    print(f'Test category predictions : {len(category_artifacts.test_query_category_map):,}')
    print(f'Category accuracy         : {category_artifacts.classifier_accuracy:.5f}')


Loading category classifier from cache: category_classifier_e3f41fe8a612f7c0.pkl
Train category predictions: 327
Test category predictions : 141
Category accuracy         : 0.92661


## Step 3: Cross-Encoder Reranker

If reranking is enabled, the notebook loads or trains the cross-encoder model from the training relevance labels.
This is a second-stage model: reorders the candidate list returned by the first stage.


In [5]:
cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=config)
if cross_encoder_reranker is None:
    print('Cross-encoder reranking disabled in config.')
else:
    print('Cross-encoder reranker is ready.')


Loading cross-encoder from cache: cross-encoder_ms-marco-MiniLM-L6-v2_c2a686d8799ae524


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder reranker is ready.


## Step 4: First-Stage Retrieval

This cell runs the retrieval stage on the **test queries**.
If category filtering is enabled, search is restricted to the predicted category; otherwise the configured final retriever searches the full corpus.


In [6]:
test_results, test_category_predictions = run_first_stage_retrieval(
    frames=frames,
    paths=paths,
    prepared_retrievers=prepared_retrievers,
    category_artifacts=category_artifacts,
    split='test',
    config=config,
)
print(f'First-stage results: {len(test_results):,} queries')


Starting category-filtered retrieval: model=embedding, top_k=12,500, queries=141, predicted_categories=5
Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_f2ec1b95e73064f5.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=22,998, queries=17, prepared_artifacts=yes, embedding_kind='queries_test_android_filtered'
Loading queries_test_android_filtered embeddings from cache: queries_test_android_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_79df14d371eb170d.npy
  [Embedding] scoring 17 queries against 22,998 docs with capped_top_k=12,500, chunk_size=32, embedding_cache_key='queries_test_android_filtered'
  [Embedding] chunk 1/1: queries 1-17
Completed retrieval: model=embedding, results=17 queries, elapsed=0.0s
Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_5c02b9494fdb73f4.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=45,301, queries=14, prepared_artifacts=yes, embedding_kind='queries_test_

## Step 5: Optional Reranking

When available, the cross-encoder rescoring step is applied to the top retrieved candidates.
The reranker can also add a category bonus when the predicted query category matches the document category.


In [7]:
test_results = rerank_retrieval_results(
    results=test_results,
    frames=frames,
    category_artifacts=category_artifacts,
    cross_encoder_reranker=cross_encoder_reranker,
    split='test',
    config=config,
)
print('Reranking step completed.')


  [CrossEncoder] reranked 141/141 queries (top_m=45, category_bonus=0.50, total_pairs=6,345)
Reranking step completed.


## Step 6: Submission Writing

The final ranked results are serialized into the exact Kaggle submission format using the sample submission schema.
The preview below is only for inspection; the important output is the CSV written to `paths.output_path`.


In [8]:
write_submission(test_results, paths, category_predictions=test_category_predictions)
print(f'Submission written to: {paths.output_path}')
submission_preview = pd.read_csv(paths.output_path)
submission_preview.head()


Submission written to: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/solutions_SeaFour.csv


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""74a05d0b-2e0f-487d-b426-8bb303e78375_109602""...",programmers
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""3fd76a85-c69c-47de-b4b6-5dedf515b0a2_30246"",...",unix
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",android
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""65a10367-e197-471c-8edc-0a73618172e1_14831"",...",unix
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""59108dd2-5f5a-4add-b947-3303c8aaa891_123331""...",tex
